In [1]:
import json
import ndjson
import os

In [11]:
# Split a large JSON file into smaller files based on maximum file size

# --- Configuration ---
# inputfile = "C:/Users/jamlung/Documents/GitHub/ocl-content/LOINC/LOINC-OCL-ETL/output-for-source-LOINC-2-81-on-staging/concepts.json"
# folder = "C:/Users/jamlung/Documents/GitHub/ocl-content/LOINC/LOINC-OCL-ETL/output-for-source-LOINC-2-81-on-staging/"
inputfile = "C:/Users/jamlung/Documents/GitHub/ocl-content/LOINC/LOINC-OCL-ETL/output-for-source-LOINC-2-81-on-staging/concepts_chunks_by_size_13.json"
folder = "C:/Users/jamlung/Documents/GitHub/ocl-content/LOINC/LOINC-OCL-ETL/output-for-source-LOINC-2-81-on-staging/"
out_file_name_root = "concepts_chunks_by_size_13_split_"
max_file_size_mb = 26  # Maximum file size in MB

# --- File Splitting Logic ---
max_file_size_bytes = max_file_size_mb * 1024 * 1024
current_output_file_index = 1
current_output_file = None
current_output_file_size = 0

print(f"Splitting '{os.path.basename(inputfile)}' into chunks of {max_file_size_mb} MB...")

with open(inputfile, 'r', encoding='utf8') as bigfile:
    for line in bigfile:
        if current_output_file is None or current_output_file_size + len(line.encode('utf-8')) > max_file_size_bytes:
            if current_output_file:
                current_output_file.close()
            
            output_filename = f"{out_file_name_root}{current_output_file_index}.json"
            output_filepath = os.path.join(folder, output_filename)
            current_output_file = open(output_filepath, 'w', encoding='utf8')
            current_output_file_size = 0
            current_output_file_index += 1
            print(f"Creating new file: {output_filename}")
        
        current_output_file.write(line)
        current_output_file_size += len(line.encode('utf-8'))

if current_output_file:
    current_output_file.close()

print("File splitting complete! ✅")

Splitting 'concepts_chunks_by_size_13.json' into chunks of 26 MB...
Creating new file: concepts_chunks_by_size_13_split_1.json
Creating new file: concepts_chunks_by_size_13_split_2.json
File splitting complete! ✅


In [2]:
# Split file by number of lines
inputfile = "C:/Users/jamlung/Documents/GitHub/ocl-content/LOINC/LOINC-OCL-ETL/output_full_02Sep2025/mappings.json"
folder = "C:/Users/jamlung/Documents/GitHub/ocl-content/LOINC/LOINC-OCL-ETL/output_full_02Sep2025/Split Mappings"
out_file_name_root = "mappings_chunks_02Sep2025_"

lines_per_file=50000


lineno=0
bigJSON = open(inputfile, 'r', encoding='utf8')
smallfile = open(folder + out_file_name_root + str(lines_per_file) + ".json", "w", encoding='utf8')

with bigJSON as bigfile:
    for line in bigfile:
        
        #replace text as needed
        # line = line.replace('LOINC-3','LOINC')
    
        lineno+=1
        if lineno % lines_per_file == 0:
            if smallfile:
                smallfile.close()
            small_filename = (folder+(out_file_name_root + '{}'.format(lineno + lines_per_file))+'.json')
            smallfile = open(small_filename, "w", encoding='utf8')
        smallfile.write(line)
    if smallfile:
        smallfile.close()

In [4]:
## Split file by lists in a JSON dictionary 

# --- Configuration for hierarchy_only.json splitting ---
inputfile = "C:/Users/jamlung/Documents/GitHub/ocl-content/LOINC/LOINC-OCL-ETL/output_full_02Sep2025/hierarchy_only.json"
folder    = "C:/Users/jamlung/Documents/GitHub/ocl-content/LOINC/LOINC-OCL-ETL/output_full_02Sep2025/Split hierarchy (5mb)/"
# inputfile = "C:/Users/jamlung/Documents/GitHub/ocl-content/LOINC/LOINC-OCL-ETL/output-arc/hierarchy_only.json"
# folder = "C:/Users/jamlung/Documents/GitHub/ocl-content/LOINC/LOINC-OCL-ETL/output-arc/"

out_file_name_root = "hierarchy_chunks_02Sep2025_"


# Choose one splitting method and comment out the other
split_by_lists = False
lists_per_file = 2 # Change this to the desired number of lists per file

split_by_size = True
max_file_size_mb = 5 # Change this to the desired maximum file size in MB

# --- File Splitting Logic for JSON Dictionaries ---

# Load the entire JSON file into a Python dictionary
with open(os.path.join(folder, inputfile), 'r', encoding='utf8') as f:
    data = json.load(f)

# Get the list of items to process
items = list(data.items())

if split_by_lists:
    print(f"Splitting '{inputfile}' by number of lists, {lists_per_file} lists per file...")
    num_files = (len(items) + lists_per_file - 1) // lists_per_file
    for i in range(num_files):
        start_index = i * lists_per_file
        end_index = min((i + 1) * lists_per_file, len(items))
        chunk = items[start_index:end_index]
        
        output_filename = f"{out_file_name_root}by_list_{i+1}.json"
        output_filepath = os.path.join(folder, output_filename)
        
        with open(output_filepath, 'w', encoding='utf8') as outfile:
            json.dump(dict(chunk), outfile, indent=2)
            
        print(f"Created file: {output_filename}")
        
elif split_by_size:
    print(f"Splitting '{inputfile}' by maximum file size, {max_file_size_mb} MB per file...")
    max_file_size_bytes = max_file_size_mb * 1024 * 1024
    
    current_output_file_index = 1
    current_output_data = {}

    for key, value in items:
        # Check size before adding the next item
        temp_data = current_output_data.copy()
        temp_data[key] = value
        
        temp_size = len(json.dumps(temp_data, indent=2).encode('utf-8'))
        
        if temp_size > max_file_size_bytes and current_output_data:
            # Write the current file
            output_filename = f"{out_file_name_root}by_size_{current_output_file_index}.json"
            output_filepath = os.path.join(folder, output_filename)
            with open(output_filepath, 'w', encoding='utf8') as outfile:
                json.dump(current_output_data, outfile, indent=2)
            print(f"Created file: {output_filename}")
            
            # Start a new file with the current item
            current_output_file_index += 1
            current_output_data = {key: value}
        else:
            current_output_data[key] = value
            
    # Write any remaining data to the last file
    if current_output_data:
        output_filename = f"{out_file_name_root}by_size_{current_output_file_index}.json"
        output_filepath = os.path.join(folder, output_filename)
        with open(output_filepath, 'w', encoding='utf8') as outfile:
            json.dump(current_output_data, outfile, indent=2)
        print(f"Created file: {output_filename}")
        
print("File splitting complete! ✅")

Splitting 'C:/Users/jamlung/Documents/GitHub/ocl-content/LOINC/LOINC-OCL-ETL/output_full_02Sep2025/hierarchy_only.json' by maximum file size, 5 MB per file...
Created file: hierarchy_chunks_02Sep2025_by_size_1.json
Created file: hierarchy_chunks_02Sep2025_by_size_2.json
Created file: hierarchy_chunks_02Sep2025_by_size_3.json
Created file: hierarchy_chunks_02Sep2025_by_size_4.json
File splitting complete! ✅
